In [52]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [53]:
data_path = Path("data/train.txt") if Path("data/train.txt").exists() else Path("nlp/data/train.txt")
df = pd.read_csv(data_path, sep=";", header=None, names=["text", "emotion"])

In [54]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [55]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [56]:
unique_emotions = df['emotion'].unique()

In [57]:
emotion_numbers = {}
i=0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i+=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [58]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [59]:
df['text'] = df['text'].apply(lambda x: x.lower())

In [60]:
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('','', string.punctuation))


In [61]:
df['text'] = df['text'].apply(remove_punc)

In [62]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new+=i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [63]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new+=i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [64]:
%pip install nltk
import nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [65]:
from nltk.corpus import stopwords  
from nltk.tokenize import word_tokenize

In [66]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abhis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abhis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abhis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [67]:
stop_words = set(stopwords.words('english'))

In [68]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [69]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if i not in stop_words:
            cleaned.append(i)

    return " ".join(cleaned)

In [70]:
df['text'] = df['text'].apply(remove)

In [71]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [72]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

In [73]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [74]:
bow_vectorizer = CountVectorizer()
x_train_bow = bow_vectorizer.fit_transform(X_train)

In [75]:
x_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116049 stored elements and shape (12800, 13359)>

In [76]:
x_test_bow = bow_vectorizer.transform(X_test)

In [77]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [78]:
nb_model  = MultinomialNB()

In [79]:
nb_model.fit(x_train_bow, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [80]:
pred_nb = nb_model.predict(x_test_bow)

In [81]:
accuracy_score(y_test, pred_nb)

0.7678125

In [82]:
tfidf_vectorizer = TfidfVectorizer()

In [83]:
x_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
x_test_tfidf = tfidf_vectorizer.transform(X_test)

In [84]:
nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [85]:
new_pred_nb = nb2_model.predict(x_test_tfidf)

In [86]:
accuracy_score(y_test, new_pred_nb)

0.6609375

In [87]:
from sklearn.linear_model import LogisticRegression

In [88]:
lg_model = LogisticRegression(max_iter=1000)

In [89]:
lg_model.fit(x_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [90]:
pred_lg = lg_model.predict(x_test_tfidf)

In [91]:
accuracy_score(y_test, pred_lg)

0.8615625

In [92]:
import joblib

model_dir = Path("models") if Path("models").exists() else Path("nlp/models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(lg_model, model_dir / "emotion_model.pkl")
joblib.dump(tfidf_vectorizer, model_dir / "tfidf_vectorizer.pkl")
joblib.dump(emotion_numbers, model_dir / "emotion_mapping.pkl")

['emotion_mapping.pkl']

In [93]:
%pip install streamlit joblib

Defaulting to user installation because normal site-packages is not writeable
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\abhis\AppData\Roaming\Python\Python311\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\abhis\AppData\Roaming\Python\Python311\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\abhis\AppData\Roaming\Python\Python311\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\abhis\AppData\Roaming\Python\Python311\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\http\client.py", line 466, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python311\Lib\s